# 第 4 章 — 拘束付き探索 (Constraints)

**ゴール**
- `sella.Constraints` で結合・角度・並進を固定する
- **緩和スキャン** で 1 次元ポテンシャル面を描き、TS のおおよその位置を見る
- 「特定の二面角だけ固定して残りを TS 最適化」のような **拘束付き鞍点探索** を体験する

題材はそのまま HCN ⇌ HNC を使います (xTB)。

In [ ]:
import numpy as np
from ase import Atoms
from tblite.ase import TBLite
from sella import Sella, Constraints

def xtb():
    return TBLite(method='GFN2-xTB', verbosity=0)

def hcn_geometry(angle_deg):
    """H-C-N の屈曲角を指定して HCN を組む。C を原点、N を x 軸正方向に置く。"""
    cn = 1.17
    ch = 1.10
    theta = np.deg2rad(180.0 - angle_deg)   # 180° = 線形 HCN
    H = [-ch * np.cos(theta), ch * np.sin(theta), 0.0]
    return Atoms('HCN', positions=[H, [0,0,0], [cn, 0, 0]])

## 緩和スキャン: H-C-N の屈曲角を 0° (線形) から 180° まで変える

各角度で `fix_angle` を入れて他の自由度だけ最小化します。鞍点付近で最大値を取るはずです。

In [ ]:
angles = np.linspace(0, 180, 19)   # 10° 刻み
energies_scan = []

for ang in angles:
    atoms = hcn_geometry(ang)
    atoms.calc = xtb()

    cons = Constraints(atoms)
    # H(0)-C(1)-N(2) の角度を ang 度に固定して残りを緩和
    cons.fix_angle((0, 1, 2), target=float(ang))

    opt = Sella(atoms, order=0, constraints=cons, logfile=None)
    opt.run(fmax=5e-3, steps=300)
    energies_scan.append(atoms.get_potential_energy())
    print(f'angle={ang:5.1f}°  E={atoms.get_potential_energy():.5f} eV')

In [ ]:
import matplotlib.pyplot as plt

E_arr = np.array(energies_scan)
E0 = E_arr.min()

plt.figure(figsize=(6, 4))
plt.plot(angles, (E_arr - E0) * 1000, marker='o')
plt.xlabel('H-C-N 屈曲角 [°] (180° = HCN 線形)')
plt.ylabel('ΔE [meV]')
plt.title('HCN ⇌ HNC 緩和スキャン')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

i_max = int(np.argmax(E_arr))
print(f'ピークは angle = {angles[i_max]:.1f}° 付近')

## 拘束付き鞍点探索

ピーク付近の構造を初期値にして、**結合長 C-N だけ固定** して TS を探してみます。  
「反応座標の他の部分は無視して、特定の自由度の範囲内で鞍点を取りたい」というときに便利な使い方です。

In [ ]:
atoms = hcn_geometry(angles[i_max])
atoms.calc = xtb()

cons = Constraints(atoms)
cons.fix_bond((1, 2), target=1.17)   # C-N 距離を 1.17 Å に固定

opt = Sella(atoms, order=1, constraints=cons, trajectory='ts_constrained.traj', logfile=None)
opt.run(fmax=1e-3, steps=500)

print(f'拘束付き TS エネルギー: {atoms.get_potential_energy():.5f} eV')
print(f'C-N 距離               : {atoms.get_distance(1, 2):.4f} Å (拘束)')
print(f'H-C-N 角度             : {atoms.get_angle(0, 1, 2):.2f}°')

## Constraints チートシート

```python
cons = Constraints(atoms)

# 並進: 原子 i を xyz 固定
cons.fix_translation(0)
cons.fix_translation(1, dim=0)              # x 方向のみ
cons.fix_translation((2, 3, 4))             # 重心を固定
cons.fix_translation()                      # 全系の重心を固定

# 結合長
cons.fix_bond((0, 1))                       # 現在値で固定
cons.fix_bond((1, 2), target=1.5)           # 値を指定
cons.fix_bond((2, 3), mic=True)             # 最小像規約

# 角度 / 二面角
cons.fix_angle((0, 1, 2), target=120)
cons.fix_dihedral((0, 1, 2, 3), target=180)
```

> **注意**: ASE の `FixAtoms` ではなく **必ず `Sella(..., constraints=cons)`** に渡してください。`atoms.constraints` に直接付けても Sella は認識しません。

## 演習

1. スキャンの刻みを `5°` まで細かくして、ピーク位置がどれくらいシャープになるか観察してください。
2. `fix_bond` を `target=1.20` にして拘束付き TS を取り直し、エネルギーと幾何の差を見てください。
3. 角度 `(0, 1, 2)` を固定したまま `order=1` で動かすと何が起きるか試してみましょう (拘束の方向が虚振動と重なって失敗するはずです)。

---
次章では同じ HCN ⇌ HNC の TS を **MACE-MP-0 (ML ポテンシャル)** で解いて、xTB の結果と比較します。